# mv alpha calculation

This notebook contains the implementation for calculating the multi-value alpha of a sample of our coded dataset.

We ran two IRR rounds on the final categorisation: one between the first and second author and the other between the first author and a research colleague. mv-alpha is calculated per category in each case.

Here's a summary of the notebook sections:
1. Install libraries and load data
2. IRR attempt 1
3. IRR attempt 2

## Install libraries

In [5]:
#install libraries here

source("irr_calculation_utils.r")
first_move_codes <- get_first_move_codes()
positive_debugging_behaviour_indicator_codes <- get_positive_debugging_indicator_codes()
added_error_codes <- get_added_error_codes()
resolved_error_codes <- get_resolved_error_codes()
inconsequential_change_codes <- get_inconsequential_change_codes()
miscellaneous_behaviour_codes <- get_miscellaneous_behaviour_codes()

The `extract_category_dataframe` function below is used for filtering each unit's code to a relevant category (used for calculating mv-alpha per category).

## IRR attempt 1
We now complete the first IRR round between the first and second author of the paper.

We calculate an mv-alpha value *per category* rather than for the whole categorisation for two main reasons:
- It gives a more granular picture of the reliability of the categorisation, which is more appropriate than a whole mv-alpha value.
- Some of the units have too many codes to calculate an mv-alpha value for the entire dataset (the `mvalpha` function won't complete).

Note that the names of some of the sub-categories here are different to the categorisation paper to ensure uniqueness (otherwise there'd be multiple codes with the same name, which could artificially inflate agreement).

In [2]:
#Load in data here

### 1. First move
Codes for this category are mutually exclusive, so we use Krippendorff's alpha rather than mv-alpha here

In [2]:
source("irr_calculation_utils.r")
first_move_dataframe <- extract_category_dataframe(df, first_move_codes)

source("mvalpha_cpp.r")
first_move_alpha <- mvalpha_cpp(first_move_dataframe, type = "nominal")
print(first_move_alpha)

Using 11 OpenMP threads
n Units: 39
n Observers: 2
n Labels: 2
n Values: 3
Max Cardinality: 1
Possible C-K pairs: 7
|C| = 0, |K| = 0, pairs = 1
|C| = 0, |K| = 1, pairs = 2, rate = 677 pairs/sec
|C| = 1, |K| = 1, pairs = 4, rate = 1908 pairs/sec

Total combinations: 7, Overall rate: 4287 pairs/sec
mvalpha (nominal): 
0.9004

### 2. Positive debugging indicators

In [ ]:
source("irr_calculation_utils.r")
positive_debugging_indicator_dataframe <- extract_category_dataframe(
    df,
    positive_debugging_indicator_codes
)

for (i in seq_len(nrow(positive_debugging_indicator_dataframe))) {
    print(positive_debugging_indicator_dataframe[i, , drop = FALSE])
}

source("mvalpha_cpp.r")
positive_debugging_indicator_alpha <- mvalpha(positive_debugging_indicator_dataframe, type = "nominal")
print(positive_debugging_indicator_alpha)

                                                 V1   V2
2 First change on line referred to in error message NULL
                                                 V1
3 First change on line referred to in error message
                                                 V2
3 First change on line referred to in error message
    V1                                                V2
4 NULL First change on line referred to in error message
                                                 V1
5 First change on line referred to in error message
                                                 V2
5 First change on line referred to in error message
                                                 V1
6 First change on line referred to in error message
                                                 V2
6 First change on line referred to in error message
                                                 V1
7 First change on line referred to in error message
                                            

ERROR: Error in aperm.default(apply(data, MARGIN = c(1, 2), function(x) {: 'perm' is of wrong length 3 (!= 2)


### 3. Added errors

On the first attempt, the `mvalpha` function did not complete.

In [ ]:
source("irr_calculation_utils.r")
added_error_dataframe <- extract_category_dataframe(df, added_error_codes)

source("mvalpha_cpp.r")
added_error_alpha <- mvalpha_cpp(added_error_dataframe, type = "nominal")
print(added_error_alpha)

: 

### 4. Resolved errors

In [ ]:
source("irr_calculation_utils.r")
resolved_error_dataframe <- extract_category_dataframe(df, resolved_error_codes)

source("mvalpha_cpp.r")
resolved_error_alpha <- mvalpha_cpp(resolved_error_dataframe, type = "nominal")
print(resolved_error_alpha)

ERROR: Error in extract_category_dataframe(df, resolved_error_codes): could not find function "extract_category_dataframe"


### 5. Inconsequential changes

In [ ]:
source("irr_calculation_utils.r")
inconsequential_change_dataframe <- extract_category_dataframe(
    df,
    inconsequential_change_codes
)

source("mvalpha_cpp.r")
inconsequential_changes_alpha <- mvalpha_cpp(inconsequential_change_dataframe, type = "nominal")
print(inconsequential_changes_alpha)

                                              V1   V2
1 Inconsequential changes to variable references NULL
                                                                                                        V1
4 Inconsequential changes to whitespace of program, Inconsequential changes to (existing) other statements
                                                                                                                                                                                      V2
4 Inconsequential changes to variable references, Inconsequential changes to whitespace of program, Inconsequential changes to (existing) other statements, Addition of unnecessary code
                                                                                                                                        V1
5 Addition/removal or editing of comments, Inconsequential changes to whitespace of program, Inconsequential changes to (existing) outputs
                              

### 6. Miscellaneous behaviours

In [ ]:
source("irr_calculation_utils.r")
miscellaneous_behaviour_dataframe <- extract_category_dataframe(
    df,
    miscellaneous_behaviour_codes
)

source("mvalpha_cpp.r")
miscellaneous_behaviour_alpha <- mvalpha_cpp(miscellaneous_behaviour_dataframe, type = "nominal")
print(miscellaneous_behaviour_alpha)

Using 11 OpenMP threads


ERROR: Error in aperm.default(apply(data, MARGIN = c(1, 2), function(x) {: 'perm' is of wrong length 3 (!= 2)


## IRR attempt 2 - pilot
This IRR attempt was conducted with a different rater (a research colleague) on four exercise attempts. We start off by loading the data (using a different approach to others).
For each column:
- Create a dataframe where each row's index is a column in the dataset
- Then for each row, populate with columns that say "yes"

In [ ]:
# Load the two pilot rater CSVs (with final row related to "any other comments" removed)
rater1_df <- read.csv("attempt_2_pilot/irr_attempt_2_rater_1_pilot.csv", stringsAsFactors = FALSE, check.names = FALSE)
rater2_df <- read.csv("attempt_2_pilot/irr_attempt_2_rater_2_pilot.csv", stringsAsFactors = FALSE, check.names = FALSE)

rater1_df <- rater1_df[-nrow(rater1_df), ]
rater2_df <- rater2_df[-nrow(rater2_df), ]

rater_codes <- data.frame(
  rater1 = I(vector("list", ncol(rater1_df) - 1)),
  rater2 = I(vector("list", ncol(rater1_df) - 1))
)

for (col_index in 2:5) {
    #Get codes where raters have categorised the behaviour as "yes"
    rater1_yes <- tolower(rater1_df[[col_index]]) == "yes"
    rater2_yes <- tolower(rater2_df[[col_index]]) == "yes"

    rater1_codes <- trimws(rater1_df[[1]][rater1_yes])
    rater2_codes <- trimws(rater2_df[[1]][rater2_yes])

    #Aggregate these codes in the rater_codes DF
    rater_codes[[1]][[col_index - 1]] <- rater1_codes
    rater_codes[[2]][[col_index - 1]] <- rater2_codes
}

write.csv(rater_codes, "attempt_2_pilot/irr_attempt_2_pilot_aggregate_codes.csv", row.names = FALSE)

### 1. First move

In [3]:
source("irr_calculation_utils.r")
first_move_dataframe <- extract_category_dataframe(rater_codes, first_move_codes)
print(first_move_dataframe)

source("mvalpha_cpp.r")
first_move_alpha <- mvalpha_cpp(first_move_dataframe, type = "nominal")
print(first_move_alpha)

                            rater1                           rater2
1 Made changes before running code Made changes before running code
2   Ran code before making changes   Ran code before making changes
3 Made changes before running code Made changes before running code
4 Made changes before running code Made changes before running code
Using 11 OpenMP threads
n Units: 4
n Observers: 2
n Labels: 2
n Values: 2
Max Cardinality: 1
Possible C-K pairs: 4
|C| = 1, |K| = 1, pairs = 4

Total combinations: 4, Overall rate: 2685 pairs/sec
mvalpha (nominal): 
1

### 2. Positive debugging behaviours

In [ ]:
source("irr_calculation_utils.r")
positive_debugging_indicator_dataframe <- extract_category_dataframe(
    rater_codes,
    positive_debugging_indicator_codes
)
print(positive_debugging_indicator_dataframe)

source("mvalpha_cpp.r")
positive_debugging_indicator_alpha <- mvalpha(positive_debugging_indicator_dataframe, type = "nominal")
print(positive_debugging_indicator_alpha)

ERROR: Error: object 'rater_codes' not found


### 3. Added errors

In [ ]:
source("irr_calculation_utils.r")
added_error_dataframe <- extract_category_dataframe(rater_codes, added_error_codes)
print(added_error_dataframe)

source("mvalpha_cpp.r")
added_error_alpha <- mvalpha_cpp(added_error_dataframe, type = "nominal",verbose = FALSE)
print(added_error_alpha)

                                                                                                                                                                                                                                                   rater1
1                                                                                                  Incorrect syntactical changes to output statements, Incorrect syntactical changes to other symbols, Incorrect semantic changes to variable assignments
2 Incorrect syntactical changes to variable assignments, Incorrect syntactical changes to other symbols, Incorrect syntactical changes involving fusion or distribution of multiple lines of code, Incorrect syntactical changes to whitespace of program
3                                                                                                  Incorrect syntactical changes to output statements, Incorrect syntactical changes to other symbols, Incorrect semantic changes to variable assignments


|C| = 0, |K| = 0, pairs = 1
|C| = 0, |K| = 1, pairs = 8, rate = 5390 pairs/sec
|C| = 0, |K| = 2, pairs = 28, rate = 28418 pairs/sec
|C| = 0, |K| = 3, pairs = 56, rate = 87202 pairs/sec
|C| = 0, |K| = 4, pairs = 70, rate = 118200 pairs/sec
|C| = 1, |K| = 1, pairs = 64, rate = 146122 pairs/sec
|C| = 1, |K| = 2, pairs = 224, rate = 182917 pairs/sec
|C| = 1, |K| = 3, pairs = 448, rate = 338359 pairs/sec
|C| = 1, |K| = 4, pairs = 560, rate = 618081 pairs/sec
|C| = 2, |K| = 2, pairs = 784, rate = 900506 pairs/sec
|C| = 2, |K| = 3, pairs = 1568, rate = 1268163 pairs/sec
|C| = 2, |K| = 4, pairs = 1960, rate = 1871164 pairs/sec
|C| = 3, |K| = 3, pairs = 3136, rate = 2613440 pairs/sec
|C| = 3, |K| = 4, pairs = 3920, rate = 3701841 pairs/sec
|C| = 4, |K| = 4, pairs = 4900, rate = 4724667 pairs/sec

Total combinations: 17727, Overall rate: 5637641 pairs/sec
NULL


### 4. Resolved errors

In [67]:
source("irr_calculation_utils.r")
resolved_error_dataframe <- extract_category_dataframe(rater_codes, resolved_error_codes)
print(resolved_error_dataframe)

source("mvalpha_cpp.r")
resolved_error_alpha <- mvalpha_cpp(resolved_error_dataframe, type = "nominal")
print(resolved_error_alpha)

                                                                                                                                                                                                                      rater1
1                                                                                                                          Resolved syntax error by changing syntax of erroneous component, Correctly resolved runtime error
2                                                                                                                                                                                            Correctly resolved syntax error
3 Correctly resolved syntax error, Resolved syntax error by changing syntax of erroneous component, Correctly resolved logical error while code not running, Correctly resolved logical error after repeated successful runs
4                           Correctly resolved runtime error, Correctly resolved type error, Correctly resolved logi

### 5. Inconsequential changes

In [68]:
source("irr_calculation_utils.r")
inconsequential_change_dataframe <- extract_category_dataframe(
    rater_codes,
    inconsequential_change_codes
)
print(inconsequential_change_dataframe)

source("mvalpha_cpp.r")
inconsequential_changes_alpha <- mvalpha_cpp(inconsequential_change_dataframe, type = "nominal")
print(inconsequential_changes_alpha)

                                                                                                                                             rater1
1                                                                                                                                              NULL
2 Inconsequential changes involving symbols, Inconsequential changes to variable references, Inconsequential changes to (existing) other statements
4                                                                                                                      Addition of unnecessary code
                                          rater2
1 Inconsequential changes to variable references
2      Inconsequential changes involving symbols
4 Inconsequential changes to variable references


Using 11 OpenMP threads
n Units: 3
n Observers: 2
n Labels: 4
n Values: 5
Max Cardinality: 3
Possible C-K pairs: 57
|C| = 0, |K| = 0, pairs = 1
|C| = 0, |K| = 1, pairs = 4, rate = 6896 pairs/sec
|C| = 0, |K| = 3, pairs = 4, rate = 19809 pairs/sec
|C| = 1, |K| = 1, pairs = 16, rate = 23622 pairs/sec
|C| = 1, |K| = 3, pairs = 16, rate = 50968 pairs/sec
|C| = 3, |K| = 3, pairs = 16, rate = 70410 pairs/sec

Total combinations: 57, Overall rate: 81790 pairs/sec
mvalpha (nominal): 
-0.0454

### 6. Miscallenous behaviours

In [69]:
source("irr_calculation_utils.r")
miscellaneous_behaviour_dataframe <- extract_category_dataframe(
    rater_codes,
    miscellaneous_behaviour_codes
)
print(miscellaneous_behaviour_dataframe)

source("mvalpha_cpp.r")
miscellaneous_behaviour_alpha <- mvalpha_cpp(miscellaneous_behaviour_dataframe, type = "nominal")
print(miscellaneous_behaviour_alpha)

                                                                                                        rater1
1                                                  Repeatedly ran erroneous program, Reverted previous changes
2 Repeatedly ran erroneous program, Repeatedly ran program at beginning of exercise, Reverted previous changes
3                                                  Repeatedly ran erroneous program, Reverted previous changes
4                                                                                    Reverted previous changes
                                                                             rater2
1                                                  Repeatedly ran erroneous program
2 Repeatedly ran erroneous program, Repeatedly ran program at beginning of exercise
3                                                  Repeatedly ran erroneous program
4                                                                              NULL
Using 11 OpenMP threads
n

## IRR attempt 2 - full

## Some old code I'm not archiving yet

In [ ]:
suppressPackageStartupMessages(library(mvalpha))
suppressPackageStartupMessages(library(magrittr))

suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(stringr))

df <- read.table("categorisation_irr_comparison.txt", sep = "\t", header = FALSE, stringsAsFactors = FALSE) # nolint
df[] <- lapply(df, function(col) lapply(col, function(x) strsplit(x, ",")[[1]]))

max_cardinality <- 6 #The max number of items a single raters' coding for a single unit can have. Both coders must have coded less than this to keep the row in the subset
#For reference, max|C| = 10 would give about a 10% sample

df_subset <- df[
    sapply(df[[1]], length) <= max_cardinality &
        sapply(df[[2]], length) <= max_cardinality,
]

print(df_subset)
cat("n =", length(df_subset[[1]]), "\n")
w <- length(unique(unlist(df_subset)))
cat("w =", w, "\n")

source("mvalpha_large.r")
alpha <- mvalpha_large(df_subset, type = "nominal", verbose = TRUE)
print(alpha)